# Route C — BERTweet (revised)

Drop-in replacement for `vu-dsp2.ipynb`. Implements the five review points plus
several runtime reductions, and produces a **deployable `.model`** in the
case-manual format.

**What changed**
1. **BERTweet-large** (`vinai/bertweet-large`) instead of `bert-base-uncased`, with the
   tokenizer's own `@USER`/`HTTPURL`/emoji normalisation (no MNTN/URL skew).
2. **Canonical `data_split.csv`** — train/val/test come from the *same* split as
   Routes A/B; the final number is reported on the **test** partition.
3. **Deployable `.model`** — a self-contained, picklable
   `{"vectorizer", "classifier"}` dict (classes live in `sentiment_deploy.py`,
   not `__main__`); labels map back to the API space **{-1, 0, 1}**.
4. **`max_length = 512`** + **dynamic padding** (no padding to a global max).
5. **Class weighting** — inverse-frequency weights that **upweight Neutral**
   (the minority + hardest class) via a custom `WeightedTrainer`.

**Extra runtime reductions** (mirroring the A/B notebooks): mixed precision
(`fp16`), length-bucketed batching (`group_by_length`), parallel tokenisation,
early stopping, larger eval batch, optional bottom-layer freezing, and
**grid search on a stratified subsample** with the final model retrained on the
full training split.

**On grid search + 5-fold CV.** For the classical Routes A/B, `GridSearchCV(cv=5)`
is cheap. For a transformer it is *not*: each fold is a full fine-tune of ~355M
parameters, so 5-fold × a grid would mean dozens of full training runs. The
applicable form here is a **validation-based grid search** (the standard for
transformer fine-tuning), implemented below. A real `StratifiedKFold` path is
included and can be switched on with `N_FOLDS > 1`, but be aware it multiplies
runtime by `N_FOLDS`.

In [8]:
# ============================ CONFIG ============================
from pathlib import Path

# --- data ---
SPLIT_CSV   = "/kaggle/input/datasets/ingvar2006/sentimentroutec/data_split.csv"   # canonical split shared with Routes A/B
TEXT_COL    = None    # None -> auto-detect among common names
LABEL_COL   = None    # None -> auto-detect; values may be {-1,0,1}, {0,1,2} or stars {1..5}
SPLIT_COL   = None    # None -> auto-detect ('split'/'set'/'partition')
TRAIN_KEYS  = {"train", "training"}
VAL_KEYS    = {"val", "valid", "validation", "dev"}
TEST_KEYS   = {"test", "testing", "holdout"}

# --- model ---
MODEL_NAME  = "vinai/bertweet-large"   # RoBERTa-large backbone; supports up to 512 tokens
MAX_LEN     = 512                      # base maxes at 128 (pos-emb 130); large allows 512 (pos-emb 514)
NUM_LABELS  = 3

# --- training ---  (sized for bertweet-large @ 512 on a single ~16GB GPU)
SEED          = 42
EPOCHS        = 4
TRAIN_BS      = 4            # large @512 is memory-heavy; drop to 2 if you OOM
GRAD_ACCUM    = 4            # effective batch = TRAIN_BS * GRAD_ACCUM = 16
EVAL_BS       = 8           # eval is cheaper but 512 tokens still adds up
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06
EARLY_STOP    = 1            # patience (epochs) on val macro-F1; 0 disables
FREEZE_BOTTOM = 0            # freeze embeddings + first N encoder layers (0 = full FT)
GRADIENT_CHECKPOINTING = True  # ~20% slower, large memory saving — needed for large@512 on a T4

# --- grid search (validation-based) ---
GRID_LR        = 2e-5   # keep as-is — LR is the one thing worth searching
GRID_WEIGHTED  = True  # was [True, False] — halves the grid (2 runs not 4)

# --- runtime ---
FP16            = True
GROUP_BY_LENGTH = True
TOKENIZE_PROC   = 4          # parallel tokenisation workers

# --- output ---
OUT_DIR     = Path("route_c_out");  OUT_DIR.mkdir(exist_ok=True)
MODEL_FILE  = "route_c_bertweet_large.model"
ROUTE_A_F1  = 0.68           # baseline for context

import random, numpy as np
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
print("Config loaded.")

Config loaded.


In [9]:
# ============================================================
# Route C (BERTweet) — Colab environment check
# Run this FIRST. It verifies GPU, libraries, model access,
# and data before you attempt the full training run.
# ============================================================
import sys, importlib, traceback

problems = []

def ok(msg):   print("  [OK]  " + msg)
def warn(msg): print("  [!!]  " + msg);
def bad(msg):  print("  [XX]  " + msg); problems.append(msg)

print("="*60)
print("1. GPU / CUDA")
print("="*60)
try:
    import torch
    if torch.cuda.is_available():
        ok(f"CUDA available — {torch.cuda.get_device_name(0)}")
        ok(f"torch {torch.__version__}, CUDA build {torch.version.cuda}")
        free, total = torch.cuda.mem_get_info()
        ok(f"GPU memory: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    else:
        bad("CUDA NOT available — you are on CPU. "
            "Runtime → Change runtime type → T4 GPU, then restart and rerun.")
except Exception as e:
    bad(f"torch import failed: {e}")

print("="*60)
print("2. Core libraries")
print("="*60)
need = ["numpy", "pandas", "transformers", "tokenizers",
        "sklearn", "bs4"]   # bs4 = BeautifulSoup (HTML cleaning)
for name in need:
    try:
        m = importlib.import_module(name)
        ver = getattr(m, "__version__", "?")
        ok(f"{name} {ver}  ({m.__file__})")
    except Exception as e:
        bad(f"{name} missing/broken: {e}")

# emoji is optional — only needed if you demojize yourself (you don't have to)
try:
    import emoji
    ok(f"emoji {emoji.__version__} (optional)")
except Exception:
    warn("emoji not installed (optional — BERTweet's normalizer handles emojis)")

print("="*60)
print("3. numpy / pandas ABI consistency")
print("="*60)
try:
    import numpy as np, pandas as pd
    # the exact operation that failed on the VU hub:
    _ = pd.DataFrame({"a": np.array([1, 2, 3])})
    ok("pandas can build a DataFrame from a numpy array (no ABI mismatch)")
    npdir = np.__file__.split("site-packages")[0]
    pddir = pd.__file__.split("site-packages")[0]
    if npdir == pddir:
        ok("numpy and pandas live in the same environment")
    else:
        bad(f"numpy ({npdir}) and pandas ({pddir}) are in DIFFERENT trees — ABI risk")
except Exception as e:
    bad(f"numpy/pandas ABI check failed: {e}")

print("="*60)
print("4. BERTweet model + tokenizer download")
print("="*60)
MODEL_NAME = "vinai/bertweet-large"
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True, use_fast=False)
    ok(f"tokenizer loaded ({MODEL_NAME})")
    # confirm the tweet-normalizer actually fires (the BERTweet advantage)
    sample = "@someone check http://x.com this is GREAT 😡"
    enc = tok.tokenize(sample)
    norm_hit = any(t in {"@USER", "HTTPURL"} for t in enc) or "@@" in " ".join(enc)
    if norm_hit or "HTTPURL" in tok.decode(tok.encode(sample)):
        ok("normalization=True is active (mentions/URLs/emoji handled by tokenizer)")
    else:
        warn("could not confirm normalizer fired — inspect tok.tokenize(sample) manually")
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    ok("model loaded with a fresh 3-class head "
       "(MISSING classifier.* keys at load time is NORMAL — those are the new head)")
    del m
except Exception as e:
    bad(f"model/tokenizer load failed: {e}")
    traceback.print_exc()

print("="*60)
print("5. Data file")
print("="*60)
# >>> EDIT this to wherever you put data_split.csv in Colab <
SPLIT_CSV = "/kaggle/input/datasets/ingvar2006/sentimentroutec/data_split.csv"
import os
try:
    if not os.path.exists(SPLIT_CSV):
        bad(f"{SPLIT_CSV} not found. Upload it (left Files panel) or mount Drive, "
            f"then set SPLIT_CSV to the right path.")
    else:
        import pandas as pd
        df = pd.read_csv(SPLIT_CSV)
        ok(f"loaded {SPLIT_CSV} — shape {df.shape}")
        ok(f"columns: {list(df.columns)}")
        # quick check for the split + label columns the notebook expects
        cols = {c.lower() for c in df.columns}
        if cols & {"split", "set", "partition", "subset", "fold"}:
            ok("a split column is present (test set will be comparable to Routes A/B)")
        else:
            warn("no split column found — Route C cell 8 will make a FRESH split "
                 "(NOT comparable to A/B). Check this before trusting the F1.")
        if cols & {"text", "review", "review_cleaned", "document", "content", "sentence"}:
            ok("a text column is present")
        else:
            warn("no obvious text column — set TEXT_COL manually in the config")
except Exception as e:
    bad(f"data check failed: {e}")

print("="*60)
print("SUMMARY")
print("="*60)
if not problems:
    print("  All critical checks passed — you're clear to run Route C.")
else:
    print(f"  {len(problems)} blocking issue(s) — fix these before training:")
    for p in problems:
        print("   - " + p)
print("="*60)

1. GPU / CUDA
  [OK]  CUDA available — Tesla T4
  [OK]  torch 2.10.0+cu128, CUDA build 12.8
  [OK]  GPU memory: 15.5 GB free / 15.6 GB total
2. Core libraries
  [OK]  numpy 2.4.6  (/usr/local/lib/python3.12/dist-packages/numpy/__init__.py)
  [OK]  pandas 2.3.3  (/usr/local/lib/python3.12/dist-packages/pandas/__init__.py)
  [OK]  transformers 5.0.0  (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)
  [OK]  tokenizers 0.22.2  (/usr/local/lib/python3.12/dist-packages/tokenizers/__init__.py)
  [OK]  sklearn 1.6.1  (/usr/local/lib/python3.12/dist-packages/sklearn/__init__.py)
  [OK]  bs4 4.13.5  (/usr/local/lib/python3.12/dist-packages/bs4/__init__.py)
  [OK]  emoji 2.15.0 (optional)
3. numpy / pandas ABI consistency
  [OK]  pandas can build a DataFrame from a numpy array (no ABI mismatch)
  [XX]  numpy (/usr/local/lib/python3.12/dist-packages/numpy/__init__.py) and pandas (/usr/local/lib/python3.12/dist-packages/pandas/__init__.py) are in DIFFERENT trees — ABI risk
4. BERT

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  [OK]  model loaded with a fresh 3-class head (MISSING classifier.* keys at load time is NORMAL — those are the new head)
5. Data file
  [OK]  loaded /kaggle/input/datasets/ingvar2006/sentimentroutec/data_split.csv — shape (255083, 6)
  [OK]  columns: ['Unnamed: 0', 'Text', 'clean_eda', 'sentiment', 'Type', 'split']
  [OK]  a split column is present (test set will be comparable to Routes A/B)
  [OK]  a text column is present
SUMMARY
  1 blocking issue(s) — fix these before training:
   - numpy (/usr/local/lib/python3.12/dist-packages/numpy/__init__.py) and pandas (/usr/local/lib/python3.12/dist-packages/pandas/__init__.py) are in DIFFERENT trees — ABI risk


## 1. Write the picklable deployment module

Writing the wrapper classes to a real `.py` file (imported below) is what makes the pickle load in the API's separate process — objects defined in a notebook live in `__main__` and fail to unpickle elsewhere. Ship this file next to `app.py`.

In [10]:
%%writefile sentiment_deploy.py
"""
sentiment_deploy.py
===================

Self-contained, picklable deployment wrapper for the Route C (BERTweet-large)
sentiment classifier, compatible with the case-manual API template.

The API loads a single ``*.model`` pickle and expects a dict::

    {"vectorizer": <obj with .transform(list[str])>,
     "classifier": <obj with .predict(X)>}

A HuggingFace transformer does not fit that interface, so this module
provides two small adapters:

* ``BertweetVectorizer`` -- a pass-through "vectorizer". It applies the SAME
  light cleaning used at training time and returns the (cleaned) strings.
  ``fit`` / ``fit_transform`` are no-ops, so the wrapper is safe even if the
  API template erroneously calls ``fit_transform`` at inference time.

* ``BertweetClassifier`` -- holds the fine-tuned weights, config and tokenizer
  files *inside the pickle* (no external paths, no dependence on a checkpoint
  directory). It rebuilds the model lazily on first use and maps the internal
  class indices {0,1,2} back to the API label space {-1, 0, 1}.

This module is model-agnostic: the config + state_dict + tokenizer are captured
dynamically from whatever model you pass in, so it serves ``vinai/bertweet-base``
and ``vinai/bertweet-large`` identically. The only large-specific default is
``max_length=512`` (base maxes out at 128 tokens; large supports up to 512).

IMPORTANT (pickle/__main__ caveat): because the API loads the pickle in a
*separate process*, the classes referenced by the pickle must be importable
there. Defining them in THIS module (not in a notebook's __main__) is what
makes the round-trip work. Ship ``sentiment_deploy.py`` alongside the API
``app.py``.
"""

from __future__ import annotations

import io
import os
import tempfile
from typing import List, Sequence

# Heavy deps (torch / transformers / bs4) are imported lazily inside methods
# so this module can be imported in lightweight contexts and unit-tested.

# Internal index -> API label.  Training uses 0=Negative, 1=Neutral, 2=Positive.
# The API/case-manual label space is -1=Negative, 0=Neutral, 1=Positive.
INDEX_TO_API_LABEL = {0: -1, 1: 0, 2: 1}

# Default max sequence length. BERTweet-large (RoBERTa-large backbone,
# max_position_embeddings=514) supports up to 512 tokens; base maxes at 128.
DEFAULT_MAX_LENGTH = 512


def normalize_text(x) -> str:
    """Light, rule-based cleaning applied identically at train and serve time.

    Only does what BERTweet's own tokenizer normalisation does NOT do: strip
    HTML (reviews contain markup) and collapse whitespace. Mention/URL/emoji
    handling is delegated to the tokenizer (``normalization=True``) so that
    train and serve stay perfectly consistent and there is no MNTN/URL skew.
    """
    if x is None:
        return ""
    x = str(x)
    if "<" in x and ">" in x:  # only pay BeautifulSoup cost when markup is likely
        try:
            from bs4 import BeautifulSoup
            x = BeautifulSoup(x, "html.parser").get_text(separator=" ")
        except Exception:
            pass
    x = " ".join(x.split())  # collapse all whitespace runs
    return x


class BertweetVectorizer:
    """Pass-through 'vectorizer' for API compatibility.

    Tokenisation happens inside the classifier, so ``transform`` just returns
    the cleaned strings. ``fit``/``fit_transform`` are no-ops -- importantly,
    ``fit_transform`` does NOT re-fit anything, so the buggy template call
    ``vectorizer.fit_transform(text)`` at inference behaves like ``transform``.
    """

    def fit(self, X=None, y=None):
        return self

    def transform(self, X: Sequence[str]) -> List[str]:
        if isinstance(X, str):
            X = [X]
        return [normalize_text(t) for t in X]

    def fit_transform(self, X: Sequence[str], y=None) -> List[str]:
        return self.transform(X)


class BertweetClassifier:
    """Self-contained, picklable BERTweet sequence classifier.

    Parameters
    ----------
    model : transformers PreTrainedModel (fine-tuned)
    tokenizer : transformers PreTrainedTokenizer
    max_length : int
        Token cap at inference. 512 for bertweet-large (default), 128 for base.
    batch_size : int
        Inference batch size. Keep modest for large on CPU (the HF free tier).
    """

    def __init__(self, model=None, tokenizer=None,
                 max_length: int = DEFAULT_MAX_LENGTH, batch_size: int = 16):
        self.max_length = int(max_length)
        self.batch_size = int(batch_size)
        self.index_to_api = dict(INDEX_TO_API_LABEL)

        # Serialised payload (populated from the live objects). Kept so the
        # pickle is fully self-contained.
        self._config = None
        self._state_dict = None       # dict[str, torch.Tensor] on CPU
        self._tokenizer_files = None  # dict[str, bytes]

        if model is not None and tokenizer is not None:
            self._capture(model, tokenizer)

        # Live objects (rebuilt lazily; never pickled).
        self._model = None
        self._tok = None

    # ---- serialisation helpers -------------------------------------------
    def _capture(self, model, tokenizer):
        """Snapshot weights/config/tokenizer into picklable payload."""
        self._config = model.config
        self._state_dict = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        with tempfile.TemporaryDirectory() as d:
            tokenizer.save_pretrained(d)
            files = {}
            for name in os.listdir(d):
                path = os.path.join(d, name)
                if os.path.isfile(path):
                    with open(path, "rb") as fh:
                        files[name] = fh.read()
            self._tokenizer_files = files

    def __getstate__(self):
        # Exclude live (non-portable) objects from the pickle.
        return {
            "max_length": self.max_length,
            "batch_size": self.batch_size,
            "index_to_api": self.index_to_api,
            "_config": self._config,
            "_state_dict": self._state_dict,
            "_tokenizer_files": self._tokenizer_files,
        }

    def __setstate__(self, state):
        self.__dict__.update(state)
        self._model = None
        self._tok = None

    # ---- lazy rebuild -----------------------------------------------------
    def _build_model(self, config, state_dict):
        """Rebuild the HF model from config + state_dict (no hub download)."""
        from transformers import AutoModelForSequenceClassification
        model = AutoModelForSequenceClassification.from_config(config)
        model.load_state_dict(state_dict)
        return model

    def _ensure(self):
        if self._model is not None:
            return
        import torch
        from transformers import AutoTokenizer

        # tokenizer
        self._tokdir = tempfile.mkdtemp(prefix="bertweet_tok_")
        for name, data in self._tokenizer_files.items():
            with open(os.path.join(self._tokdir, name), "wb") as fh:
                fh.write(data)
        self._tok = AutoTokenizer.from_pretrained(
            self._tokdir, normalization=True, use_fast=False)

        # model
        model = self._build_model(self._config, self._state_dict)
        self._device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(self._device)
        model.eval()
        self._model = model

    # ---- inference --------------------------------------------------------
    def predict(self, X: Sequence[str]):
        """Return a list of API labels in {-1, 0, 1} for the input texts.

        Accepts either raw strings or strings already passed through the
        vectorizer (cleaning is idempotent, so both work).
        """
        import numpy as np
        import torch

        if isinstance(X, str):
            X = [X]
        texts = [normalize_text(t) for t in X]
        self._ensure()

        preds = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self._tok(batch, max_length=self.max_length, truncation=True,
                            padding=True, return_tensors="pt")
            enc = {k: v.to(self._device) for k, v in enc.items()}
            with torch.no_grad():
                logits = self._model(**enc).logits
            idx = logits.argmax(dim=1).cpu().numpy()
            preds.extend(int(self.index_to_api[int(j)]) for j in idx)
        return preds

    # convenience
    def predict_proba(self, X: Sequence[str]):
        import torch
        import torch.nn.functional as F
        if isinstance(X, str):
            X = [X]
        texts = [normalize_text(t) for t in X]
        self._ensure()
        out = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self._tok(batch, max_length=self.max_length, truncation=True,
                            padding=True, return_tensors="pt")
            enc = {k: v.to(self._device) for k, v in enc.items()}
            with torch.no_grad():
                logits = self._model(**enc).logits
            out.append(F.softmax(logits, dim=1).cpu().numpy())
        return out and __import__("numpy").vstack(out)


Writing sentiment_deploy.py


In [11]:
import importlib, sentiment_deploy
importlib.reload(sentiment_deploy)
from sentiment_deploy import (
    normalize_text, BertweetVectorizer, BertweetClassifier, INDEX_TO_API_LABEL,
)
print("Imported wrapper. internal->API label map:", INDEX_TO_API_LABEL)

Imported wrapper. internal->API label map: {0: -1, 1: 0, 2: 1}


## 2. Helpers — label normalisation, column detection, class weights, metric

In [12]:
import pandas as pd, numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, classification_report

def _find_col(cols, candidates):
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    return None

def normalize_labels_to_3(series):
    # Map any of {-1,0,1} / {0,1,2} / star ratings {1..5} -> {0,1,2}
    # where 0=Negative, 1=Neutral, 2=Positive.
    s = pd.to_numeric(series, errors="coerce")
    u = set(int(v) for v in s.dropna().unique())
    if u <= {-1, 0, 1}:
        return s.map({-1: 0, 0: 1, 1: 2}).astype("int64")
    if u <= {0, 1, 2}:
        return s.astype("int64")
    if u <= {1, 2, 3, 4, 5}:
        return s.map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2}).astype("int64")
    raise ValueError(f"Unrecognised label set: {u}")

def class_weights(y):
    w = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=np.asarray(y))
    return w.astype("float32")   # neutral (minority) gets the largest weight

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1_macro": f1_score(labels, preds, average="macro"),
            "accuracy": accuracy_score(labels, preds)}
print("Helpers ready.")

Helpers ready.


## 3. Load the canonical split

Reads `data_split.csv` (the same artifact Routes A/B use), applies the light rule-based cleaning, and slices train/val/test. If the file has no split column, it falls back to a stratified split with a clear warning.

In [13]:
raw = pd.read_csv(SPLIT_CSV)
print("Loaded", SPLIT_CSV, "shape", raw.shape, "| columns:", list(raw.columns))

text_col  = TEXT_COL  or _find_col(raw.columns, ["text","review_cleaned","review","document","content","sentence"])
label_col = LABEL_COL or _find_col(raw.columns, ["sentiment_encoded","label","sentiment","target","y","numericvalue"])
split_col = SPLIT_COL or _find_col(raw.columns, ["split","set","partition","subset","fold"])
assert text_col and label_col, f"Could not find text/label columns in {list(raw.columns)}"
print(f"Using text='{text_col}', label='{label_col}', split='{split_col}'")

raw["_text"]  = raw[text_col].astype(str).map(normalize_text)
raw["labels"] = normalize_labels_to_3(raw[label_col])
raw = raw.dropna(subset=["labels"]).reset_index(drop=True)
raw["labels"] = raw["labels"].astype("int64")

if split_col is not None:
    s = raw[split_col].astype(str).str.lower().str.strip()
    train_df = raw[s.isin(TRAIN_KEYS)].copy()
    val_df   = raw[s.isin(VAL_KEYS)].copy()
    test_df  = raw[s.isin(TEST_KEYS)].copy()
    if len(val_df) == 0:               # some splits only have train/test
        from sklearn.model_selection import train_test_split
        train_df, val_df = train_test_split(train_df, test_size=0.1,
            stratify=train_df["labels"], random_state=SEED)
        print("No val partition found -> carved 10% val out of train.")
else:
    print("WARNING: no split column found -> making a fresh stratified 80/10/10 split. "
          "Results will NOT be comparable to Routes A/B unless this matches their split.")
    from sklearn.model_selection import train_test_split
    train_df, tmp = train_test_split(raw, test_size=0.2, stratify=raw["labels"], random_state=SEED)
    val_df, test_df = train_test_split(tmp, test_size=0.5, stratify=tmp["labels"], random_state=SEED)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    dist = d["labels"].value_counts().sort_index().to_dict()
    print(f"{name:5s}: {len(d):>7d}  dist(0/1/2)={dist}")

Loaded /kaggle/input/datasets/ingvar2006/sentimentroutec/data_split.csv shape (255083, 6) | columns: ['Unnamed: 0', 'Text', 'clean_eda', 'sentiment', 'Type', 'split']
Using text='Text', label='sentiment', split='split'
train:  178557  dist(0/1/2)={0: 62851, 1: 44461, 2: 71245}
val  :   38263  dist(0/1/2)={0: 13468, 1: 9528, 2: 15267}
test :   38263  dist(0/1/2)={0: 13468, 1: 9528, 2: 15267}


## 4. Tokeniser + datasets (dynamic padding)

`normalization=True` lets BERTweet handle mentions/URLs/emojis. We tokenise once (parallelised) and pad **per batch** via `DataCollatorWithPadding` — no fixed-width padding to MAX_LEN (512).

In [14]:
from transformers import AutoTokenizer, DataCollatorWithPadding
import torch
from torch.utils.data import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True, use_fast=False)

class TokenizedDataset(Dataset):
    """Plain torch Dataset of pre-tokenized (unpadded) examples.

    Avoids the `datasets`/`pyarrow` dependency. Padding is done per batch by
    DataCollatorWithPadding. Supports ds["labels"] column access so the rest of
    the notebook (class weights, test labels) works unchanged.
    """
    def __init__(self, texts, labels, tokenizer, max_len, chunk=1000):
        ids, masks = [], []
        texts = list(texts)
        for i in range(0, len(texts), chunk):
            b = tokenizer(texts[i:i + chunk], truncation=True, max_length=max_len)
            ids.extend(b["input_ids"]); masks.extend(b["attention_mask"])
        self.input_ids = ids
        self.attention_mask = masks
        self.labels = [int(x) for x in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        if isinstance(idx, str):                      # ds["labels"] -> whole column
            return getattr(self, "labels" if idx in ("label", "labels") else idx)
        return {"input_ids": self.input_ids[idx],
                "attention_mask": self.attention_mask[idx],
                "labels": self.labels[idx]}

def encode(df):
    return TokenizedDataset(df["_text"].tolist(), df["labels"].tolist(),
                            tokenizer, MAX_LEN)

train_ds_full = encode(train_df)
val_ds        = encode(val_df)
test_ds       = encode(test_df)
collator      = DataCollatorWithPadding(tokenizer=tokenizer)
print("Tokenised.", len(train_ds_full), "train examples | example lengths:",
      [len(train_ds_full[i]['input_ids']) for i in range(3)])

Tokenised. 178557 train examples | example lengths: [13, 22, 23]


## 5. Weighted trainer + training routine

`WeightedTrainer` injects the inverse-frequency class weights into the loss so Neutral is upweighted. `TrainingArguments` is built in a version-robust way (the `evaluation_strategy`/`eval_strategy` rename differs across `transformers` versions).

In [15]:
import inspect, torch, torch.nn as nn
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback)
import inspect
_TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._cw = (torch.tensor(class_weight, dtype=torch.float)
                    if class_weight is not None else None)
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self._cw.to(logits.device) if self._cw is not None else None
        loss = nn.CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def make_args(output_dir, lr, epochs, do_eval=True):
    common = dict(
        output_dir=str(output_dir), learning_rate=lr, num_train_epochs=epochs,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO, seed=SEED,
        fp16=FP16, group_by_length=GROUP_BY_LENGTH, logging_steps=100,
        report_to="none", save_total_limit=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
    )
    # the eval-strategy arg was renamed across versions
    eval_key = ("eval_strategy" if "eval_strategy" in _TA_PARAMS else
                "evaluation_strategy" if "evaluation_strategy" in _TA_PARAMS else None)
    if eval_key:
        common[eval_key] = "epoch" if do_eval else "no"
    if do_eval and EARLY_STOP and eval_key:
        common.update(save_strategy="epoch", load_best_model_at_end=True,
                      metric_for_best_model="f1_macro", greater_is_better=True)
    # drop any kwargs this transformers version doesn't accept (e.g. group_by_length)
    dropped = [k for k in common if k not in _TA_PARAMS]
    common = {k: v for k, v in common.items() if k in _TA_PARAMS}
    if dropped:
        print("note: TrainingArguments version doesn't support", dropped, "- skipped")
    return TrainingArguments(**common)

def maybe_freeze(model, n):
    if not n:
        return
    base = model.base_model
    for p in base.embeddings.parameters():
        p.requires_grad = False
    layers = base.encoder.layer
    for layer in layers[:n]:
        for p in layer.parameters():
            p.requires_grad = False
    print(f"Froze embeddings + first {n} encoder layers.")

def new_model():
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
    maybe_freeze(m, FREEZE_BOTTOM)
    return m

def train_eval(train_ds, eval_ds, lr, epochs, weighted, tag="run"):
    cw = class_weights(train_ds["labels"]) if weighted else None
    args = make_args(OUT_DIR / tag, lr, epochs, do_eval=eval_ds is not None)
    cbs = ([EarlyStoppingCallback(EARLY_STOP)]
           if (eval_ds is not None and EARLY_STOP) else None)
    trainer = WeightedTrainer(
        model=new_model(), args=args, class_weight=cw,
        train_dataset=train_ds, eval_dataset=eval_ds,
        data_collator=collator, compute_metrics=compute_metrics, callbacks=cbs)
    trainer.train()
    metrics = trainer.evaluate() if eval_ds is not None else {}
    return trainer, metrics
print("Trainer ready.")

Trainer ready.


## 7. Retrain best config on full train → evaluate on TEST

In [16]:
# ============================================================================
# Route C (BERTweet-large) — FINAL: train-once -> eval -> package -> download
# Self-contained. No grid search: fixed lr=2e-5, weighted=True.
# Skips training if `final_model` is in memory OR a checkpoint is on disk.
# ============================================================================
import glob, os, gc, pickle, numpy as np, torch
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import f1_score, classification_report
from IPython.display import FileLink, display

# --- fixed config (LR tuning skipped) ---------------------------------------
BEST_LR       = 2e-5
BEST_WEIGHTED = True
FINAL_DIR     = OUT_DIR / "final"     # train_eval(..., tag="final") writes here

def find_checkpoint(base):
    """Loadable model dir under `base`, or None. Prefers a clean saved model
    (config.json at base), else the highest-step Trainer checkpoint-* subdir."""
    base = Path(base)
    if (base / "config.json").exists():
        return str(base)
    cks = sorted(glob.glob(str(base / "checkpoint-*")),
                 key=lambda p: int(p.split("-")[-1]))
    return cks[-1] if cks else None

@torch.no_grad()
def infer_logits(model, df, bs=EVAL_BS):
    """Plain batched forward pass -> logits (n, 3). Mirrors the serve-time path."""
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    texts = df["_text"].tolist()                # already normalize_text()-cleaned
    out = []
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i+bs], max_length=MAX_LEN, truncation=True,
                        padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        out.append(model(**enc).logits.detach().cpu().numpy())
    return np.vstack(out).astype(np.float64)

# --- load-if-saved, else train (the expensive step) -------------------------
trainer = None
ckpt = find_checkpoint(FINAL_DIR)
if globals().get("final_model") is not None:
    print("final_model already in memory -> reusing (no retrain, no reload).")
elif ckpt:
    print(f"-> loading existing checkpoint: {ckpt}  (NO retrain)")
    final_model = AutoModelForSequenceClassification.from_pretrained(
        ckpt, num_labels=NUM_LABELS)
else:
    print(f"-> no checkpoint under {FINAL_DIR} -> TRAINING "
          f"(lr={BEST_LR}, weighted={BEST_WEIGHTED}, epochs={EPOCHS})")
    trainer, _ = train_eval(train_ds_full, val_ds, BEST_LR, EPOCHS,
                            BEST_WEIGHTED, tag="final")
    final_model = trainer.model
    trainer.save_model(str(FINAL_DIR))          # clean config+weights for next run
    tokenizer.save_pretrained(str(FINAL_DIR))
    print(f"   saved -> {FINAL_DIR}  (re-running this cell will now skip training)")

# --- evaluate on TEST -------------------------------------------------------
test_true = np.array(test_ds["labels"])
test_pred = np.argmax(infer_logits(final_model, test_df), axis=1)
test_f1   = f1_score(test_true, test_pred, average="macro")
print(f"\n=== ROUTE C (BERTweet-large) TEST macro-F1 = {test_f1:.4f} "
      f"({test_f1*100:.1f}%) | Route A baseline = {ROUTE_A_F1:.2f} ===\n")
print(classification_report(test_true, test_pred,
      target_names=["Negative(0)", "Neutral(1)", "Positive(2)"], digits=3))

# --- package the deployable .model ------------------------------------------
final_model.to("cpu").eval()
deployment = {
    "vectorizer": BertweetVectorizer(),
    "classifier": BertweetClassifier(model=final_model, tokenizer=tokenizer,
                                     max_length=MAX_LEN, batch_size=EVAL_BS),
}
with open(MODEL_FILE, "wb") as fh:
    pickle.dump(deployment, fh)
print(f"Wrote {MODEL_FILE} ({os.path.getsize(MODEL_FILE)/1e6:.1f} MB)")

# --- round-trip test exactly as the API will do it --------------------------
with open(MODEL_FILE, "rb") as fh:
    loaded = pickle.load(fh)
sample = ["I liked the product.", "I did not like it.", "It's okay, nothing special."]
y = loaded["classifier"].predict(loaded["vectorizer"].transform(sample))
print("round-trip predictions:", y)
assert all(v in (-1, 0, 1) for v in y), "labels must be in the API space {-1,0,1}"
print("Round-trip OK — labels in {-1,0,1}.")

# --- click-to-download link (works on Kaggle/Jupyter and Colab) -------------
try:
    from google.colab import files as _colab_files     # Colab: triggers browser download
    print("\nColab detected — starting download...")
    _colab_files.download(MODEL_FILE)
except Exception:
    print("\nClick to download your .model file:")
    display(FileLink(MODEL_FILE))                       # Kaggle/Jupyter: clickable link

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


-> no checkpoint under route_c_out/final -> TRAINING (lr=2e-05, weighted=True, epochs=4)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,2.267742,0.555219,0.756120,0.777069
2,2.004080,0.555633,0.766273,0.781172
3,1.630294,0.587732,0.763682,0.784596


...

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   saved -> route_c_out/final  (re-running this cell will now skip training)

=== ROUTE C (BERTweet-large) TEST macro-F1 = 0.7644 (76.4%) | Route A baseline = 0.68 ===

              precision    recall  f1-score   support

 Negative(0)      0.818     0.826     0.822     13468
  Neutral(1)      0.571     0.660     0.612      9528
 Positive(2)      0.909     0.813     0.859     15267

    accuracy                          0.780     38263
   macro avg      0.766     0.766     0.764     38263
weighted avg      0.793     0.780     0.784     38263

Wrote route_c_bertweet_large.model (1425.1 MB)
round-trip predictions: [1, -1, 0]
Round-trip OK — labels in {-1,0,1}.

Colab detected — starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Package the deployable `.model`

Builds the `{"vectorizer", "classifier"}` dict, pickles it, then **reloads and predicts** to confirm the round-trip works and that outputs are in the API label space **{-1, 0, 1}**.

In [17]:
import pickle

model_for_deploy = trainer.model            # Trainer already unwraps DataParallel
model_for_deploy.to("cpu").eval()

deployment = {
    "vectorizer": BertweetVectorizer(),
    "classifier": BertweetClassifier(model=model_for_deploy, tokenizer=tokenizer,
                                     max_length=MAX_LEN, batch_size=EVAL_BS),
}
with open(MODEL_FILE, "wb") as fh:
    pickle.dump(deployment, fh)
import os
print(f"Wrote {MODEL_FILE} ({os.path.getsize(MODEL_FILE)/1e6:.1f} MB)")

# --- round-trip test exactly as the API will do it ---
with open(MODEL_FILE, "rb") as fh:
    loaded = pickle.load(fh)

sample = ["I liked the product.", "I did not like it.", "It's okay, nothing special."]
X = loaded["vectorizer"].transform(sample)          # API: vectorizer.transform(...)
y = loaded["classifier"].predict(X)                 # API: classifier.predict(...)
print("round-trip predictions:", y)
assert all(v in (-1, 0, 1) for v in y), "labels must be in the API space {-1,0,1}"
print("Round-trip OK — labels in {-1,0,1}.")

Wrote route_c_bertweet_large.model (1425.1 MB)
round-trip predictions: [1, -1, 0]
Round-trip OK — labels in {-1,0,1}.


## 9. Sanity check

In [18]:
tests = [
    "This product is fantastic! Everyone should buy this.",
    "Disappointed with my purchase, it stopped working after one day.",
    "It's okay, but I was hoping it would be better.",
    "@user the new update is HTTPURL totally broke my app 😡",
]
labels = {-1: "Negative", 0: "Neutral", 1: "Positive"}
for t, p in zip(tests, loaded["classifier"].predict(tests)):
    print(f"[{labels[p]:>8}] {t}")

[Positive] This product is fantastic! Everyone should buy this.
[Negative] Disappointed with my purchase, it stopped working after one day.
[ Neutral] It's okay, but I was hoping it would be better.
[Negative] @user the new update is HTTPURL totally broke my app 😡
